# Second classification run

The classification is repeated with the same model at temperature 0.1, this time asking it to label
the utterances **without** splitting them: the utterances of the first run are sent back as a CSV
block and the model annotates each row. `04_agreement_metrics` compares the two runs.

* **Input:** `labeled_conversations/<coding scheme>.csv`, prompts in `coding_schemes/<scheme>/no_segmentation/`
* **Output:** `validation/llm_rerun/<coding scheme>.csv`
* **Check:** the second run covers 95-99% of the utterances of the first one

As in `01_few_shot_classification`, `RUN_API = False` reads the run made for the study instead of
calling the API.

In [2]:
import pandas as pd
from pathlib import Path
import os
import json
from io import StringIO

# Dataset to work on: 'data' is the study data, 'data_example' the synthetic example.
DATA = Path('..') / os.environ.get('DATASET_FOLDER', 'data')

CODING_SCHEMES = ['lin_dialogue_acts',
                  'acosta_epistemic_stance',
                  'del_bosque_cma']

RUN_API = False

LABEL_COLUMN = 'label'

encodings = {}
# Read dataframe
for coding_scheme in CODING_SCHEMES:
    df = pd.read_csv(DATA / 'labeled_conversations' / (coding_scheme + '.csv'))

    encodings[coding_scheme] = df

# Variables

**Should be modified for each run**

In [3]:
CODING_SCHEME = 'lin_dialogue_acts'
OUT_FILE = CODING_SCHEME + '.csv'
OUT_FOLDER = 'validation'

encodings[CODING_SCHEME].head()

,activity_id,user_id,role,message_num,utterance_num,sentence_en,label,type
0,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,model,0,0,"Hello! 😸 I am SIMBA, and I will help you think...",Greeting,Statement
1,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,model,0,1,What are the 2 to 5 main ideas to remember fro...,Open Question,Question
2,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,model,0,2,What are the points of the course that still s...,Open Question,Question
3,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,model,0,3,I am here to accompany you in this reflection! 🌟,Reassurance,Statement
4,asst_ncmAPAoB3MT4TIBrhhRAjpJ9,s108,user,1,0,Can you clarify the difference between an isol...,Information Question,Question


## Send the conversation as a CSV block

One row per utterance, so that the model has nothing to segment.

In [4]:
df = encodings[CODING_SCHEME].copy()

df['message_num'] = df['message_num'].astype(int)
df['utterance_num'] = df['utterance_num'].astype(int)

conversations_text = []
for (activity_id, user_id), group in df.groupby(['activity_id', 'user_id'], sort=False):
    group = group.sort_values(['message_num', 'utterance_num'])

    # Get all column names except the grouping columns
    columns = [col for col in group.columns if col not in ['activity_id', 'user_id', 'label', 'type']]

    conversation_lines = [",".join(columns)]  # Header row

    for _, row in group.iterrows():
        # Always escape quotes and wrap in quotes for sentence_en
        text = str(row['sentence_en']).replace('"', '""')
        conversation_lines.append(f"{row['role']},{row['message_num']},{row['utterance_num']},\"{text}\"")

    conversation_csv = "\n".join(conversation_lines)

    conversations_text.append({
        'activity': activity_id,
        'user': user_id,
        'conversation_csv': conversation_csv
    })


# To access the first conversation as CSV text
conversations_text[0]

{'activity': 'asst_ncmAPAoB3MT4TIBrhhRAjpJ9',
 'user': 's108',
 'conversation_csv': 'role,message_num,utterance_num,sentence_en\nmodel,0,0,"Hello! 😸 I am SIMBA, and I will help you think about the following questions:"\nmodel,0,1,"What are the 2 to 5 main ideas to remember from the course entitled \'Transformations and first principle\'?"\nmodel,0,2,"What are the points of the course that still seem obscure to you?"\nmodel,0,3,"I am here to accompany you in this reflection! 🌟"\nuser,1,0,"Can you clarify the difference between an isolated and open system"\nmodel,2,0,"Sure! 🌟"\nmodel,2,1,"1. **Isolated system**: It cannot exchange matter or energy with its environment."\nmodel,2,2,"For example, a well-sealed box in which no energy (heat or work) can enter or exit."\nmodel,2,3,"2. **Open system**: It can exchange both matter and energy with its environment."\nmodel,2,4,"For example, a water reservoir where water can enter and exit, and where heat can be transferred."\nmodel,2,5,"The key i

## Functions for prompt reading and classification

Same functions as in `01_few_shot_classification`, with two differences: the prompts come from the
`no_segmentation` folder, and the temperature is 0.1 with the response forced to JSON.

In [5]:
from openai import OpenAI
from tqdm import tqdm


def read_prompts(folder: str):

    path = Path('..') / "coding_schemes" / folder / "no_segmentation"

    PROMPT_SYSTEM_NAME = path / 'prompt_system.txt'
    PROMPT_USER_NAME = path / 'prompt_user.txt'
    PROMPT_AGENT_NAME = path / 'prompt_agent.txt'

    with open(PROMPT_SYSTEM_NAME) as f:
        PROMPT_SYSTEM = f.read()

    with open(PROMPT_USER_NAME) as f:
        PROMPT_USER = f.read()

    with open(PROMPT_AGENT_NAME) as f:
        PROMPT_AGENT = f.read()

    return (PROMPT_SYSTEM, PROMPT_USER, PROMPT_AGENT)

errors = []
def classify_sentences(client: OpenAI,
                       message: str,
                       prompt_system: str,
                       prompt_user: str,
                       prompt_agent: str):

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": prompt_system},
            {"role": "user", "content": prompt_user},
            {"role": "assistant", "content": prompt_agent},
            {"role": "user", "content": message}
        ],
        temperature=0.1,  # Set 0.1
        stream=False,
        timeout=1500  # Set the timeout to 1500 seconds
    )

    # handle errors
    try:
        return response.choices[0].message.content
    except:
        errors.append({'message': message, 'response': response})
        return None


def classify_conversation(openai_client: OpenAI,
                          prompt_system: str,
                          prompt_user: str,
                          prompt_agent: str,
                          conversations: list):
    out = []
    for conversation in tqdm(conversations, desc='Classifying...'):
        response = classify_sentences(openai_client,
                                      str(conversation),
                                      prompt_system,
                                      prompt_user,
                                      prompt_agent)
        out.append(response)
    return out

## Parse the responses

Entries that cannot be parsed are collected instead of stopping the run; in the original notebook they
were written to a file (`lin_dialogue_acts_error_act_4.json`, …) and repaired by hand.

In [6]:
def process_json_entries(json_entries: list, label_column: str):

    all_dataframes = []
    error_entries = []

    for i, entry in enumerate(json_entries):
        try:
            # Parse JSON if it's a string
            data = json.loads(entry) if isinstance(entry, str) else entry

            # Extract the main fields
            activity = data['activity']
            user = data['user']
            conversation_csv = data['conversation_csv']

            # Read CSV data into dataframe
            csv_df = pd.read_csv(StringIO(conversation_csv))

            # Add the activity and user columns
            csv_df['activity_id'] = activity
            csv_df['user_id'] = user

            # Reorder columns
            csv_df = csv_df[['activity_id', 'user_id', 'role', 'message_num', 'utterance_num',
                             'sentence_en', 'annotation']]

            all_dataframes.append(csv_df)

        except Exception as e:
            error_entries.append({'index': i, 'error': str(e), 'entry': entry})
            print(f'Error at entry {i}: {e}')

    print(f'Processing complete. Successful: {len(all_dataframes)}, Errors: {len(error_entries)}')

    combined_df = pd.concat(all_dataframes, ignore_index=True)
    combined_df.rename(columns={'annotation': label_column}, inplace=True)

    return combined_df, error_entries

## Run the classification

One coding scheme at a time, as it was run; the loop covers the three schemes when the labels are read
from the files of the study.

In [7]:
from dotenv import load_dotenv

path_out = DATA / OUT_FOLDER / 'llm_rerun'
path_out.mkdir(parents=True, exist_ok=True)

if RUN_API:
    # Load the API KEY
    load_dotenv(Path('..') / '.env')

    OPENAI_KEY = os.getenv("OPENAI_API_KEY")

    openai_client = OpenAI(api_key=OPENAI_KEY)

    prompt_system, prompt_user, prompt_agent = read_prompts(CODING_SCHEME)

    labeled_data = classify_conversation(openai_client,
                                         prompt_system,
                                         prompt_user,
                                         prompt_agent,
                                         conversations_text)

    out_file, error_entries = process_json_entries(labeled_data, LABEL_COLUMN)

    out_file.to_csv(path_out / OUT_FILE, index=False)

In [8]:
for coding_scheme in CODING_SCHEMES:
    rerun = pd.read_csv(path_out / (coding_scheme + '.csv'))
    first_run = encodings[coding_scheme]

    print(coding_scheme, ':', len(rerun), 'utterances in the second run',
          '({:.2f}% of the first run)'.format(100 * len(rerun) / len(first_run)))

lin_dialogue_acts : 4658 utterances in the second run (95.51% of the first run)
acosta_epistemic_stance : 4882 utterances in the second run (99.07% of the first run)
del_bosque_cma : 4922 utterances in the second run (99.17% of the first run)
